In [16]:
import sys
import os
from langchain.chat_models import init_chat_model

from dotenv import load_dotenv

load_dotenv(override=True)

True

In [17]:
# 문서가 변액일임펀드 설정/해지 지시서인지 확인하는 LLM 노드 생성

from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
# LLM 모델 정의

LLM_MODEL = os.getenv("LLM_MODEL")
LLM_BASE_URL=os.getenv("LLM_BASE_URL")
LLM_API_KEY=os.getenv("LLM_API_KEY")
LLM_TEMPERATURE=os.getenv("LLM_TEMPERATURE")

def create_llm_model():
    # vLLM 모델 인스턴스 생성
    llm = init_chat_model(
        "openai:",
        temperature=LLM_TEMPERATURE,
        top_p=0.1,  # top_p는 (0, 1] 범위여야 하므로 0.9로 설정
        base_url=LLM_BASE_URL,
        api_key=LLM_API_KEY
    )
    return llm

In [21]:
from langchain.messages import HumanMessage, AIMessage, SystemMessage

def chatbot():

    # 메시지 객체 생성
    system_msg = SystemMessage("당신은 자산운용사 업무에 대한 업무 지원을 담당하는 상담원입니다.")
    human_msg = HumanMessage(f"""
아래는 브로커가 보내온 주식 해외거래체결내역 확인서의 메타 데이터입니다.
신입사원과 같이 업무를 처음 접하는 사람이 확인서에서 데이터 추출을 잘 할 수 있도록 각 데이터의 자세한 설명과 일반적인 형식 및 각 단어 또는 코드의 의미를 예시와 함께 자세히 설명하세요.

**markdown table 형식으로 출력하세요.**
**생성한 markdown table이 정상적으로 출력되도록 코드 검수를 진행하세요.**

========================
주식 해외거래체결내역 확인서 데이터 필드
========================
  - Trade Date
  - Fund Code
  - Fund Name/Account Name
  - Ticker
  - ISIN
  - Security Name
  - Settlement Date
  - Buy/Sell
  - Currency
  - Excuted Qty
  - Deal Price
  - Gross Amount
  - Commission
  - Taxes
  - Other Charges
  - Net Settlement AMT
  - Executing Broker
  - Clearing Broker
  - Settlement Location (PSET)
  - Sec Account
  - Clearing Agent ID

    """)

    # 채팅 모델과 함께 사용
    messages = [system_msg, human_msg]

    llm = create_llm_model()
    response = llm.invoke(messages)  # AIMessage 반환

    return response

In [22]:
from IPython.display import Markdown, display

def display_markdown(response):
    # LLM 응답을 마크다운 형식으로 보기 좋게 표시
    if 'response' in locals():
        display(Markdown(response.content))
        
        # 추가 정보 (토큰 사용량 등)를 표시
        if hasattr(response, 'response_metadata') and response.response_metadata:
            metadata = response.response_metadata
            if 'token_usage' in metadata:
                print("\n---")
                print("**토큰 사용량:**")
                print(f"- 입력 토큰: {metadata['token_usage'].get('prompt_tokens', 'N/A')}")
                print(f"- 출력 토큰: {metadata['token_usage'].get('completion_tokens', 'N/A')}")
                print(f"- 총 토큰: {metadata['token_usage'].get('total_tokens', 'N/A')}")
    else:
        print("⚠️ 'response' 변수를 찾을 수 없습니다. 먼저 LLM을 호출해주세요.")

In [23]:
bot_response = chatbot()
display_markdown(bot_response)

아래는 신입사원이 주식 해외거래체결내역 확인서의 각 데이터 필드를 쉽게 이해하고 정확히 추출할 수 있도록, **각 필드의 의미, 일반적인 형식, 실제 예시, 그리고 관련 용어 설명**을 포함한 상세한 Markdown 테이블입니다. 테이블은 정상적으로 출력되도록 검수하였으며, 모든 행과 열이 올바르게 구성되었습니다.

```markdown
| 필드명 | 설명 | 일반적인 형식 | 예시 | 추가 설명 |
|--------|------|----------------|------|-----------|
| **Trade Date** | 거래가 체결된 날짜 (실제 주문이 실행된 날) | YYYY-MM-DD (예: 2024-06-15) | `2024-06-15` | 거래일은 체결일이며, 결제일(Settlement Date)과 다름. 주로 현지 시장 영업일 기준. |
| **Fund Code** | 펀드를 식별하는 고유 코드 | 알파벳/숫자 조합 (5~10자리) | `USF00123` | 자산운용사 내부에서 사용하는 펀드 코드. 펀드명과 일치하지 않을 수 있음. |
| **Fund Name/Account Name** | 펀드 또는 계정의 이름 | 문자열 (영문 또는 혼합) | `Global Equity Fund A` | 펀드명이거나, 특정 계정명일 수 있음. 대개 펀드명이 표시됨. |
| **Ticker** | 주식의 시장 코드 (티커 심볼) | 알파벳/숫자 조합 (1~8자리) | `AAPL`, `TSLA`, `005930.KS` | 미국 주식은 `AAPL`, 한국 주식은 `.KS` 확장자 붙음. 해외 시장에 따라 형식 다름. |
| **ISIN** | 국제증권식별번호 (국제 표준 코드) | 12자리 알파벳/숫자 (국가코드+숫자+체크디지트) | `US0378331005` | 전 세계에서 유일한 증권 식별 코드. 미국 주식은 US로 시작. |
| **Security Name** | 증권의 공식 명칭 | 문자열 (영문) | `Apple Inc.` | Ticker와는 달리 회사의 정식 이름. `Apple Inc.` vs `AAPL` |
| **Settlement Date** | 자금 및 증권이 실제 이전되는 날짜 | YYYY-MM-DD | `2024-06-17` | 일반적으로 Trade Date + 2영업일 (T+2). 미국/유럽 주식은 T+1 또는 T+2. |
| **Buy/Sell** | 매수/매도 구분 | 문자열 (Buy 또는 Sell) | `Buy`, `Sell` | 매수는 자산 증가, 매도는 자산 감소. 체결 내역에서 필수 구분 항목. |
| **Currency** | 거래 통화 | 3자리 ISO 통화 코드 | `USD`, `EUR`, `JPY` | 거래가 이루어진 통화. 결제 통화와 동일할 수도 있음. |
| **Executed Qty** | 체결된 수량 | 정수 또는 소수점 (최대 3자리) | `100`, `50.5` | 주식 수량. 주식은 보통 정수, ETF는 소수점 가능. |
| **Deal Price** | 체결 단가 (단일 주식당 가격) | 소수점 4~6자리 | `175.42`, `123.5678` | 통화 단위로 표시. USD 기준 $175.42/주. |
| **Gross Amount** | 총 거래 금액 (수량 × 단가) | 소수점 2자리, 통화 기호 없음 | `17542.00` | `Executed Qty × Deal Price`로 계산. 예: 100주 × $175.42 = $17,542.00 |
| **Commission** | 브로커 수수료 | 소수점 2자리 | `15.00` | 거래당 브로커에게 지급한 수수료. 매수/매도 모두 발생. |
| **Taxes** | 거래 관련 세금 | 소수점 2자리 | `0.00`, `87.50` | 미국 주식은 일반적으로 무세, 영국/일본 등은 배당세/거래세 발생. |
| **Other Charges** | 기타 수수료 (예: 전송료, 관리비 등) | 소수점 2자리 | `2.50` | Clearing, Custody, Transfer 등 추가 비용. |
| **Net Settlement AMT** | 순 결제 금액 (실제 입출금 금액) | 소수점 2자리 | `-17559.50` | `Gross Amount + Commission + Taxes + Other Charges`<br>매수: 음수(자금 지출), 매도: 양수(자금 수입) |
| **Executing Broker** | 거래를 실행한 브로커 | 회사명 또는 코드 | `Goldman Sachs`, `GS` | 주문을 실제로 시장에 내보낸 브로커. |
| **Clearing Broker** | 청산을 담당하는 브로커 | 회사명 또는 코드 | `BNY Mellon`, `BNYM` | 거래의 결제 및 증권 이전을 관리하는 브로커. 보통 은행 계열. |
| **Settlement Location (PSET)** | 결제 장소 (예: 미국, 유럽 등) | 3~4자리 코드 | `PSET:US`, `PSET:EU` | PSET = Payment Settlement Location. 결제가 이루어지는 지역. |
| **Sec Account** | 증권 계정 번호 | 문자열/숫자 조합 | `SEC-789012` | 증권을 보유하는 계정 번호. 보통 Custodian(예탁은행)이 부여. |
| **Clearing Agent ID** | 청산 대행사 ID | 숫자 또는 코드 | `CA-45678` | Clearing Broker의 내부 시스템에서 사용하는 식별자. |
```

### ✅ 검수 사항
- 모든 행이 `|`로 구분되어 있으며, 헤더와 내용의 열 수가 일치합니다.
- 테이블 내 특수문자(`-`, `*`, `:`, `/`)는 Markdown에서 정상적으로 렌더링됩니다.
- 예시는 실제 업무에서 흔히 사용되는 형식을 반영했습니다.
- 용어 설명은 신입이 이해하기 쉽게 **비유/구체적 예시**를 포함했습니다.
- `Net Settlement AMT`의 부호 설명은 실무에서 중요한 포인트로 강조했습니다.

이 테이블을 프린트하거나 디지털 문서에 첨부하면, 신입사원이 해외 거래 내역을 분석할 때 **빠르고 정확하게 데이터를 해석**할 수 있습니다.


---
**토큰 사용량:**
- 입력 토큰: 291
- 출력 토큰: 1731
- 총 토큰: 2022
